In [1]:
from src.helpers.model_matrix import build_model_matrix_from_wrds

tickers_list = [
    'AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA', 'META',
    # 'GOOGL', 'AVGO', 'GOOG', 'LLY', 'WMT', 'JPM', 'SPY', 'BRK', 'IVV',
    # 'VOO', 'V', 'MA', 'XOM', 'ORCL', 'UNH', 'VTI', 'COST', 'PG', 'HD', 'NFLX', 'BRK', 'JNJ', 'BAC', 'CRM', 'QQQ','ABBV',
    # 'KO', 'CVX', 'TMUS', 'MRK', 'CSCO', 'WFC', 'ACN', 'NOW', 'TSM', 'AXP', 'PEP', 'MCD', 'IBM', 'MS', 'DIS', 'LIN',
    # 'TMO', 'ABT', 'AMD', 'ADBE', 'PM', 'ISRG', 'GE', 'GS', 'INTU', 'CAT', 'TXN', 'QCOM', 'RY', 'VZ', 'DHR', 'BKNG', 'T', 'PLTR',
    # 'BLK', 'VUG', 'SPGI', 'RTX', 'PFE', 'NEE', 'HON', 'CMCSA', 'PGR', 'AMGN', 'LOW', 'ANET', 'UNP', 'SYK', 'TJX', 'VEA',
    # 'C', 'AMAT', 'BA', 'SCHW', 'BSX', 'KKR', 'ETN', 'SHOP', 'COP', 'VTV', 'UBER', 'BX', 'BND', 'AGG', 'PANW', 'ADP',
    # 'IEFA', 'FI'
]

all_stocks = build_model_matrix_from_wrds(
    wrds_user="jbernatchez",
    start="2016-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    tickers=tickers_list,
    use_run="last"  # "new", "last", or a specific folder name (i.e. "run_20250914_133747")
)

[info] Using run folder: run_20250923_174828 (reuse=True)
[info] Reuse mode: all required Parquet files are present. No extraction performed.
{'run_folder': 'wrds_extracts\\run_20250923_174828', 'reuse': True, 'artifacts': {'dsf.parquet': 'wrds_extracts\\run_20250923_174828\\dsf.parquet', 'stocknames.parquet': 'wrds_extracts\\run_20250923_174828\\stocknames.parquet', 'ff.parquet': 'wrds_extracts\\run_20250923_174828\\ff.parquet', 'ibes_stats.parquet': 'wrds_extracts\\run_20250923_174828\\ibes_stats.parquet', 'ibes_act.parquet': 'wrds_extracts\\run_20250923_174828\\ibes_act.parquet'}}
[info] Removed 0 permnos(companies) for having zero in cfacpr or cfacshr
[info] Removed 0 permnos(companies) for exceeding the threshold of negative prices
[info] ibes_stats: 274,942 (official_ticker, stat_date) pairs have >1 row (multiple horizons/periodicities). Will collapse before join.
[info] df_prices(+ibes): 95.2% missing in n_analysts.
[info] df_prices(+ibes): 95.2% missing in cons_mean.
[info] df_

In [2]:
from src.helpers.model_matrix import align_and_fill_dates_across_tickers

df = align_and_fill_dates_across_tickers(all_stocks=all_stocks)

All groups have consistent date indices and 1162 rows each.
(5855, 41)
                column  n_null  pct_null  has_nulls
0               ticker       0       0.0      False
1        adjclose_lead       0       0.0      False
2           adj_mktcap       0       0.0      False
3                  vol       0       0.0      False
4                 retx       0       0.0      False
5           n_analysts       0       0.0      False
6                 n_up       0       0.0      False
7               n_down       0       0.0      False
8            cons_mean       0       0.0      False
9          cons_median       0       0.0      False
10          cons_stdev       0       0.0      False
11           cons_high       0       0.0      False
12            cons_low       0       0.0      False
13             cons_cv       0       0.0      False
14      cons_range_pct       0       0.0      False
15       adjclose_lag0       0       0.0      False
16       adjclose_lag1       0       0.0     

In [3]:
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

random.seed(42)

tau_label = 0.0  # Buy/Sell/Hold labeling threshold on next-day log-return adjclose_lead
tau_buy = 0.05  # SCORE > tau_buy  => BUY
tau_sell = -0.05  # SCORE < tau_sell => SELL
gamma = 1.0  # weight scaling (position size = gamma * SCORE), clip later
w_max = 0.20  # per-name weight cap (abs)
fixedWindow = True  # True = sliding window, False = expanding window
initial = 145
horizon = 40  # approx 2m validation per window
step = 20  # approx 1 month step

assert df.index.names == ["permno", "date"]
dates_all = df.index.get_level_values("date").unique().sort_values()
print(len(dates_all))
assert len(dates_all) >= initial + horizon, "Not enough dates for the rolling scheme."

1162


In [4]:
# Build multi-class target DIR: {-1, 0, +1}
# adjclose_lead already equals next-day log return(t -> t+1)
DIR = pd.Series(0, index=df.index, dtype=int)
DIR.loc[df["adjclose_lead"] > tau_label] = +1
DIR.loc[df["adjclose_lead"] < -tau_label] = -1

# drop ticker, adjclose_lead and date-like / id (already in index)
cat_cols = [c for c in ["act_measure_lag1", "pdicity_lag1"] if c in df.columns]  # strings
num_cols = [c for c in df.columns if c not in (["ticker", "adjclose_lead"] + cat_cols)]

In [5]:
# Model: multinomial logistic with standardization (numeric) + one-hot (categorical)
ct = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols)
], remainder="drop", sparse_threshold=1.0)

clf = LogisticRegression(
    solver="saga",
    max_iter=2000,
    n_jobs=-1,
    C=1.0
)

pipe = Pipeline([("prep", ct), ("clf", clf)])

In [6]:
# Rolling windows (time-slice)
pred_score = pd.Series(index=df.index, dtype=float)  # SCORE = p(+1) - p(-1)
pred_dir = pd.Series(index=df.index, dtype=int)  # argmax-based class (for confusion matrix)
used_mask = pd.Series(False, index=df.index)  # rows that got validated in some window

start_pos = 0
while start_pos + initial + horizon <= len(dates_all):
    train_dates = dates_all[start_pos: start_pos + initial]
    valid_dates = dates_all[start_pos + initial: start_pos + initial + horizon]

    if fixedWindow:
        start_pos += step
    else:
        # expanding: grow initial, slide validation
        start_pos += step  # still move start of validation forward by step; training grows implicitly

    # Slice rows
    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_cols + cat_cols]
    y_tr = DIR.loc[idx_tr]

    X_va = df.loc[idx_va, num_cols + cat_cols]

    # Fit & predict
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)  # columns ordered like classes_ => [-1, 0, +1]

    # Map to SCORE and predicted direction
    clses = pipe.named_steps["clf"].classes_
    col_m1 = int(np.where(clses == -1)[0][0]) if -1 in clses else None
    col_0 = int(np.where(clses == 0)[0][0]) if 0 in clses else None
    col_p1 = int(np.where(clses == +1)[0][0]) if +1 in clses else None

    p_up = proba[:, col_p1] if col_p1 is not None else 0
    p_dn = proba[:, col_m1] if col_m1 is not None else 0

    score = p_up - p_dn
    yhat = clses[np.argmax(proba, axis=1)]

    # Store
    pred_score.loc[idx_va] = score
    pred_dir.loc[idx_va] = yhat
    used_mask.loc[idx_va] = True

c:\Users\justi\OneDrive\Documentos\HEC Montreal\1st semester\Empirical Finance\Term Assignment\MATH60610A-portfolio-backtesting\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\justi\OneDrive\Documentos\HEC Montreal\1st semester\Empirical Finance\Term Assignment\MATH60610A-portfolio-backtesting\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [7]:
# Convert scores to signals & portfolio weights
signal = pd.Series(0, index=df.index, dtype=int)
signal.loc[pred_score > tau_buy] = +1
signal.loc[pred_score < tau_sell] = -1

weights = (gamma * pred_score).clip(lower=-w_max, upper=+w_max)  # per-name weight at time t

In [8]:
# Portfolio daily returns and equity curve (only on validated rows)
# r_t = sum_i w_{i,t} * adjclose_lead_{i,t}; here adjclose_lead equals next-day log return

# sum across names at each date.
valid_df = df.loc[used_mask].copy()
valid_df["score"] = pred_score.loc[used_mask]
valid_df["signal"] = signal.loc[used_mask]
valid_df["weight"] = weights.loc[used_mask]

# Per-date portfolio log-return
port_ret_by_date = (valid_df["weight"] * valid_df["adjclose_lead"]).groupby(level="date").sum()
equity = np.exp(port_ret_by_date.cumsum())
equity.index.name = "date"

In [10]:
# Classification metrics on validation set
true_dir_val = DIR.loc[used_mask]
pred_dir_val = pred_dir.loc[used_mask]

cm = confusion_matrix(true_dir_val, pred_dir_val, labels=[-1, 0, +1])
report = classification_report(
    true_dir_val, pred_dir_val,
    labels=[-1, 0, +1],
    target_names=["Down", "Flat", "Up"],
    zero_division=0
)

rf_series = df.loc[used_mask, "rf"].groupby(level="date").mean()
excess_ret = port_ret_by_date - rf_series

ann_factor = 252.0
mu_ann = excess_ret.mean() * ann_factor
sd_ann = excess_ret.std(ddof=0) * np.sqrt(ann_factor)
sharpe = mu_ann / sd_ann if sd_ann > 0 else np.nan

roll_max = equity.cummax()
dd = 1.0 - equity / roll_max
max_dd = dd.max()

In [11]:
print("=== Validation coverage ===")
print(
    f"Validated dates: {port_ret_by_date.index.min().date()} .. {port_ret_by_date.index.max().date()}  (#days={len(port_ret_by_date)})")
print(f"Validated rows:  {used_mask.sum()} of {len(df)}")

print("\n=== Confusion Matrix (rows=true, cols=pred) ===")
print(pd.DataFrame(cm, index=["Down", "Flat", "Up"], columns=["Down", "Flat", "Up"]))

print("\n=== Classification Report ===")
print(report)

print("=== Portfolio Stats (validation) ===")
print(f"Ann. excess return: {mu_ann: .4f}")
print(f"Ann. vol (excess):  {sd_ann: .4f}")
print(f"Sharpe (excess):    {sharpe: .3f}")
print(f"Max drawdown:       {max_dd: .3%}")

print("\n=== Equity curve (head) ===")
print(equity.head())
print("\n=== Equity curve (tail) ===")
print(equity.tail())

=== Validation coverage ===
Validated dates: 2016-12-15 .. 2020-12-04  (#days=1000)
Validated rows:  5000 of 5810

=== Confusion Matrix (rows=true, cols=pred) ===
      Down  Flat    Up
Down   961     1  1287
Flat     4     0    11
Up    1125     6  1605

=== Classification Report ===
              precision    recall  f1-score   support

        Down       0.46      0.43      0.44      2249
        Flat       0.00      0.00      0.00        15
          Up       0.55      0.59      0.57      2736

    accuracy                           0.51      5000
   macro avg       0.34      0.34      0.34      5000
weighted avg       0.51      0.51      0.51      5000

=== Portfolio Stats (validation) ===
Ann. excess return:  0.1357
Ann. vol (excess):   0.3846
Sharpe (excess):     0.353
Max drawdown:        34.294%

=== Equity curve (head) ===
date
2016-12-15    0.999994
2016-12-16    0.993880
2016-12-19    0.993381
2016-12-20    0.995779
2016-12-21    0.998977
dtype: float64

=== Equity curve (t

In [12]:
from src.helpers._extract import ensure_dir
import quantstats as qs

# Strategy simple returns (datetime index)
strategy_rets = equity.pct_change().dropna()
strategy_rets.name = "YaTS"

# Collapse factors to daily Series (date index)
rf_by_date = (
    df.reset_index()[["date", "rf"]]
    .dropna()
    .groupby("date", as_index=True)["rf"]
    .mean()
    .astype(float)
    .sort_index()
)

mktrf_by_date = (
    df.reset_index()[["date", "mktrf"]]
    .dropna()
    .groupby("date", as_index=True)["mktrf"]
    .mean()
    .astype(float)
    .sort_index()
)

# align all rows to strategy dates
rf_series = rf_by_date.reindex(strategy_rets.index).ffill().bfill()
bench_rets = (mktrf_by_date + rf_by_date).reindex(strategy_rets.index).ffill().bfill()
rf_series.name = "RiskFree"
bench_rets.name = "Market"

# (encountered bug: can't pass a time-series to QS, so we convert to excess returns and then pass rf=0.0 to QS)
strategy_excess = (strategy_rets - rf_series).dropna()
bench_excess = (bench_rets - rf_series).reindex(strategy_excess.index).dropna()

# ensure both series share the exact same dates
common_idx = strategy_excess.index.intersection(bench_excess.index)
strategy_excess = strategy_excess.reindex(common_idx)
bench_excess = bench_excess.reindex(common_idx)

ensure_dir("out")
qs.reports.html(
    strategy_excess,
    benchmark=bench_excess.to_frame("Market"),
    rf=0.0,
    periods_per_year=252,
    output="out/yats_tearsheet.html",
    title="Yet another Trading Strategy (YaTS)"
)